# XD-Violence Normal (A) Pull + Validation

Pulls a deterministic, class-balanced sample of Normal (A-labeled) clips
from XD-Violence via the same public HuggingFace mirror used for the
riot pull (`xdviolence_riot_pull.ipynb`) -- same access path, same
validation pattern.

**Why capped, not the full Normal class:** the full Normal class is
2346 clips / 53.4 GiB (verified via the HF tree API before writing
this) -- 5x the riot class's 9.78 GiB and badly imbalanced against it
(485 riot clips). This pulls a fixed, seeded random sample capped at
`TARGET_COUNT` (default 485, matching the riot class 1:1) so the two
classes are usable together for supervised training without needing
class-weighting as a first pass. Raise `TARGET_COUNT` later if you
want a deliberately imbalanced set.

## Step 0 — Setup

In [ ]:
!pip install requests -q

In [ ]:
import csv
import os
import random
import re
import urllib.parse

import requests
import cv2
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

FINAL_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_normal'
os.makedirs(FINAL_DIR, exist_ok=True)
print('Saving to:', FINAL_DIR)

SEED = 42  # matches src/config.py SEED, for consistency across this project
TARGET_COUNT = 485  # matched 1:1 to the riot class pulled in xdviolence_riot_pull.ipynb

## Step 1 — List the HF repo and build the Normal-clip manifest

Filters for a *pure* Normal label (just `A`, no co-occurring codes) --
unlike the riot filter, which deliberately included combined labels
like `B4-G-0`, a clip should not be treated as "Normal" if it also
carries any other content label.

In [ ]:
REPO = "jherng/xd-violence"
DIRS = {
    "data/video/1-1004": "train",
    "data/video/1005-2004": "train",
    "data/video/2005-2804": "train",
    "data/video/2805-3319": "train",
    "data/video/3320-3954": "train",
    "data/video/test_videos": "test",
}
BASE_DL = f"https://huggingface.co/datasets/{REPO}/resolve/main/"


def is_pure_normal(path: str) -> bool:
    """True only if the *entire* label field is 'A' -- no co-occurring
    codes. A clip labeled e.g. 'A-B2' is not a clean negative example."""
    m = re.search(r'_label_([A-Za-z0-9\-]+?)(?:-\d+-\d+)?\.\w+$', path)
    if not m:
        return False
    return m.group(1).split('-') == ['A']


def list_dir(dirpath: str):
    url = f"https://huggingface.co/api/datasets/{REPO}/tree/main/{urllib.parse.quote(dirpath)}?limit=1000"
    out = []
    while url:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        out.extend(r.json())
        link = r.headers.get("Link", "")
        m = re.search(r'<([^>]+)>;\s*rel="next"', link)
        url = m.group(1) if m else None
    return out

In [ ]:
all_normal_rows = []
for d, split in DIRS.items():
    print(f"Listing {d} ...")
    files = list_dir(d)
    n_normal = 0
    for f in files:
        if f.get("type") != "file" or not is_pure_normal(f["path"]):
            continue
        path = f["path"]
        vid_id = path.rsplit("/", 1)[-1]
        download_url = BASE_DL + urllib.parse.quote(path).replace("%2F", "/")
        all_normal_rows.append({
            "id": vid_id, "label": "normal", "split": split,
            "size_bytes": f.get("size", 0), "url": download_url,
        })
        n_normal += 1
    print(f"  {len(files)} files total, {n_normal} pure-Normal (A)")

total_gib = sum(r["size_bytes"] for r in all_normal_rows) / (1024 ** 3)
print(f"\nFound {len(all_normal_rows)} pure-Normal clips, {total_gib:.2f} GiB total (full class).")

In [ ]:
# Deterministic, seeded sample -- same seed always yields the same clips,
# regardless of machine/session (mirrors src/config.py's set_seed pattern).
random.seed(SEED)
manifest_rows = random.sample(all_normal_rows, k=min(TARGET_COUNT, len(all_normal_rows)))

sample_gib = sum(r["size_bytes"] for r in manifest_rows) / (1024 ** 3)
print(f"Sampled {len(manifest_rows)} clips ({sample_gib:.2f} GiB) out of {len(all_normal_rows)} available.")

manifest_path = os.path.join(FINAL_DIR, "manifest_xdviolence_normal.csv")
with open(manifest_path, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["id", "label", "split", "size_bytes", "url"])
    w.writeheader()
    w.writerows(manifest_rows)
print(f"Manifest saved to {manifest_path}")

df = pd.DataFrame(manifest_rows)
df.groupby("split").agg(clips=("id", "count"), gib=("size_bytes", lambda s: s.sum() / 1024**3))

## Step 2 — Dry run: download + validate 5 clips first

In [ ]:
def download_one(row, dest_dir):
    dest = os.path.join(dest_dir, row["id"])
    if os.path.exists(dest) and os.path.getsize(dest) == row["size_bytes"]:
        return True
    try:
        with requests.get(row["url"], stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    fh.write(chunk)
        return True
    except Exception as e:
        print(f"  FAILED {row['id']}: {e}")
        if os.path.exists(dest):
            os.remove(dest)
        return False


def validate_clip(row, dest_dir):
    path = os.path.join(dest_dir, row["id"])
    result = {"id": row["id"], "exists": os.path.exists(path)}
    if not result["exists"]:
        return result
    result["size_on_disk"] = os.path.getsize(path)
    result["size_matches"] = result["size_on_disk"] == row["size_bytes"]
    cap = cv2.VideoCapture(path)
    result["opens"] = cap.isOpened()
    if result["opens"]:
        ok, _frame = cap.read()
        result["first_frame_readable"] = ok
    cap.release()
    return result


dry_rows = manifest_rows[:5]
dry_ok = [r for r in dry_rows if download_one(r, FINAL_DIR)]
dry_validation = pd.DataFrame([validate_clip(r, FINAL_DIR) for r in dry_ok])
dry_validation

In [ ]:
assert dry_validation["exists"].all(), "Some dry-run files are missing on disk"
assert dry_validation["size_matches"].all(), "Some dry-run files have the wrong size -- truncated download"
assert dry_validation["opens"].all(), "Some dry-run files won't open in OpenCV -- corrupt download"
assert dry_validation["first_frame_readable"].all(), "Some dry-run files opened but have no readable frames"
print("Dry run fully validated -- safe to pull the rest.")

## Step 3 — Download the rest (resumable)

In [ ]:
remaining = [r for r in manifest_rows if r not in dry_rows]
print(f"Downloading {len(remaining)} remaining Normal clips...")
ok_count = 0
failed = []
for i, row in enumerate(remaining, 1):
    if download_one(row, FINAL_DIR):
        ok_count += 1
    else:
        failed.append(row["id"])
    if i % 25 == 0 or i == len(remaining):
        print(f"  {i}/{len(remaining)} processed ({ok_count} ok, {len(failed)} failed)")

print(f"\nDone. {ok_count}/{len(remaining)} downloaded.")
if failed:
    print("Failed IDs (re-run this cell to retry):")
    for fid in failed:
        print(" ", fid)

## Step 4 — Validate the full pull

In [ ]:
full_validation = pd.DataFrame([validate_clip(r, FINAL_DIR) for r in manifest_rows])
n = len(full_validation)
print(f"{n} clips in manifest")
print(f"  exists on disk:      {full_validation['exists'].sum()}/{n}")
print(f"  size matches:        {full_validation['size_matches'].sum()}/{n}")
print(f"  opens in OpenCV:     {full_validation['opens'].sum()}/{n}")
print(f"  first frame readable:{full_validation['first_frame_readable'].sum()}/{n}")

bad = full_validation[~(full_validation.get('exists', False)
                        & full_validation.get('size_matches', False)
                        & full_validation.get('opens', False)
                        & full_validation.get('first_frame_readable', False))]
if len(bad):
    print(f"\n{len(bad)} clip(s) need attention:")
    display(bad)
else:
    print("\nAll Normal clips downloaded and validated clean.")

## Next steps

- Riot clips: `My Drive/CVPHG/xdviolence_riot/` (485 clips).
- Normal clips: `My Drive/CVPHG/xdviolence_normal/` (this notebook,
  up to 485 clips).
- Both are now in place for `feature_extraction_fusion.ipynb` to run
  against and for the MLP classifier notebook after that.
- The third class ("Protest", peaceful/pre-escalation) still has **no
  identified data source** -- confirmed the Kaggle
  `jpmiller/protests-against-police-violence` dataset can't supply it
  (744/748 files are Capitol riot footage, the rest are non-video
  metadata). This blocks the 3-class target in the fusion architecture
  until a source is found -- binary Riot/Normal is what's actually
  buildable right now.